# Session 12 · Local and open models
## Ollama, open weights, quantization, and measuring your own laptop

**How to use this notebook.** It has three parts.

| Part | What it is | Code? |
|---|---|---|
| **Part A · Read before you run** | Why this topic matters, the words you need, and how today's session is built | No code. Read it first (about 15 minutes). |
| **Part B · Hands-on** | Sections 1 to 6. Every section follows the same six steps (below). | Yes |
| **Part C · Wrap-up** | Summary, interview questions, troubleshooting | No |

Every section in Part B follows the same six steps:

1. **What and why** — what we are about to do and why it matters.
2. **The idea** — the concept, with an analogy and often a small diagram.
3. **What the next cell does** — a numbered walk-through *before* you see the code.
4. **The code** — run it.
5. **What you should see** — how to read the output, and what it means if yours is different.
6. **Check yourself** — one short question and a common mistake to avoid.

Boxes marked **▸ Read more** are optional. Click to open them when you want the deeper background; you can skip them in class.

**Before starting:** `git pull`, `uv sync` (no new packages). Best with Ollama running and two local models (`llama3.2:3b`, `qwen2.5:7b`). **No Ollama, no key? It still runs top to bottom**: model cells say "skipped"; the rest works offline.

---
# Part A · Read before you run

## A1. Why this session matters

### The problem, in one story
A bank's credit team reads about two thousand loan files a day: salary slips, bank statements, ID documents. They want an assistant that extracts income and existing EMIs and drafts a first assessment. Compliance has one rule: **this customer data must not leave the bank's network.** A hosted API is out, so the team runs an **open-weight model** on its own machines.

Four things go wrong in the first two weeks:

1. **The model does not fit.** An engineer loads a 20-billion-parameter model in full 16-bit form on a 16 GB laptop: about 40 GB of numbers. The laptop freezes. Nobody did the one-line memory sum.
2. **The licence was never read.** The best model was **gated**; someone accepted its licence on a personal account. Legal asks who agreed on behalf of the bank, and whether commercial use is allowed. The project pauses.
3. **The agent stops being an agent.** The prototype **called tools** on a hosted model. On the small local model, the same request returns a polite sentence, *"I would look up account 00123456 for you"*, and no tool call. The code waited for a structured request and got prose.
4. **"It will be as fast as the cloud."** It is not. For overnight batch reading that is fine; for a call-centre screen that must answer in three seconds, it is not.

None of these is a reason to avoid local models. Each is a thing you can **check in five minutes before you commit**. That is today's topic.

### What you learn today, and what each one prevents

| You learn to... | It prevents... | The everyday name |
|---|---|---|
| Say where an open model comes from, under which names and licence | Using weights you may not use, or the wrong model | Read the label before you buy |
| Compute memory as **parameters × bytes per parameter** | A model that cannot fit, or oversized hardware | Will the sofa fit through the door? |
| Explain **quantization** and its trade-off | Believing 4-bit is free, or a smaller model | Squeezing the numbers |
| Test whether a local model really **calls tools** | An agent built on a model that only talks | Can it press the button? |
| **Measure** TTFT and tokens per second | Promising a speed you never measured | Measure, don't guess |

### Where this is used in industry
- **Banking and insurance (BFS):** document reading, policy search and code assistants on data that must stay inside the bank, served from the bank's own GPU servers with the same OpenAI-style API you use today.
- **Healthcare:** summarising clinical notes that contain patient health information, without sending them to a third party. Session 45's showcase runs a healthcare graph on a local model for this reason.
- **Manufacturing and field work:** assistants on a factory floor or remote site with weak internet; a small quantized model keeps working offline.
- **Every engineering team:** a free, private sandbox, and the fallback when a free tier hits its rate limit.

### Why interviewers care
- *"When would you choose a self-hosted open-weight model over a hosted API?"*
- *"What is quantization, and what do you lose?"*
- *"Roughly how much memory does a 7-billion-parameter model need?"*
- *"Is an open-weight model the same as open source?"*
- *"How would you check that a model supports tool calling before you build an agent on it?"*

Part C has model answers: a definition, an example from this lab, one honest limit.

**Words you will hear:** *local* = your own computer; *self-hosted* = machines your organisation controls (same open weights, bigger hardware); *on-premises* = inside the organisation's own building. In all three, **the prompt never goes to a model provider.**

## A2. Where this session fits in the course

```
 S10  The model landscape ─────► the method: ELIMINATE (hard rules) → SCORE → VERIFY;
  │                              data residency as a hard rule; first memory sum
 S11  Multi-provider calls ────► one client, many providers: only base_url, key, model change
  │                              (Ollama is just a provider at localhost:11434)
 S12  Local and open models ───► TODAY: where open weights come from, quantization sizing,
  │                              tool-calling probe, local vs hosted speed MEASURED
 S13  Multimodal + structured ─► images in, guaranteed JSON out (closes Module 1)
  │
 S14  Prompt anatomy ──────────► Module 2 starts; your work repository begins
  │
 S20  Function calling ────────► the full tool-call protocol: today's probe, done properly
 S34  What is an agent ────────► an agent = a loop of tool calls; a model that cannot call
  │                              tools cannot be the brain of one
 S45  Showcase C3 ─────────────► a healthcare graph with PHI redaction, run on a local model
 S57  Responsible AI ──────────► licences, model cards and data residency as governance
```

**What you need from earlier sessions:**
- A `.env` with your Groq key (Sessions 2, 11); without it, the Groq half of section 5 says "skipped".
- **Ollama** with `llama3.2:3b` pulled (Session 9 homework). Check: `ollama list`.
- From Session 10: the **memory sum**, `time_stream()`, the checkable EMI prompt, and the **three jobs**: public FAQ bot, loan-document reader (customer data, must stay in-house), call-centre agent-assist (answer within 3 seconds).

## A3. The words you need today

Read these once now; each appears in bold when first used in Part B.

---
**1. Open-weight model**
- **Definition:** a model whose trained numbers (the **weights**) are published, so anyone the licence allows can download and run them.
- **Analogy:** a published recipe: you can cook it at home instead of only eating at the restaurant.
- **Example:** Llama 3.2, Qwen 2.5 and gpt-oss are open-weight; a frontier vendor's flagship is only reachable over its API.
- **Often confused with:** *open source*. Training data and code are often not released, and the licence may limit commercial use.
- **Comes back in:** S45, S57.

---
**2. Hugging Face**
- **Definition:** the public hub where most open-weight models, datasets and tokenizers are published.
- **Analogy:** a public library for models, as GitHub is for code.
- **Example:** `meta-llama/Llama-3.2-3B-Instruct` (*organisation / model*).
- **Often confused with:** a model provider. It publishes the files; today's prompts do not go there.
- **Comes back in:** S24 (choosing an embedding model).

---
**3. Model card**
- **Definition:** the page that describes a model: training, intended use, licence, known limits.
- **Analogy:** the leaflet inside a medicine box.
- **Example:** it tells you whether commercial use is allowed and which languages were tested.
- **Comes back in:** S57.

---
**4. Gated model**
- **Definition:** a model you can download only after requesting access and accepting its licence.
- **Analogy:** a library book you must sign for at the desk.
- **Example:** the Meta Llama pages on Hugging Face say "request access".
- **Often confused with:** a paid model. Gating is about terms; most open-weight models are free to download.
- **Comes back in:** S57.

---
**5. Ollama**
- **Definition:** a free tool that downloads open-weight models in a laptop-friendly format and serves them at `http://localhost:11434` with an OpenAI-compatible API.
- **Analogy:** a capsule coffee machine: it does not grow the coffee, it makes brewing at home easy.
- **Example:** `ollama pull qwen2.5:7b`, then the same Python client you use for Groq talks to it.
- **Often confused with:** a model. Ollama trains nothing; it **packages and runs** models published elsewhere.
- **Comes back in:** S21 (swap providers), S45.

---
**6. Ollama tag**
- **Definition:** Ollama's short name for a model: *family*`:`*size or variant*.
- **Analogy:** a nickname next to a passport name (Hugging Face).
- **Example:** `meta-llama/Llama-3.2-3B-Instruct` → `llama3.2:3b`.
- **Comes back in:** every local-model session.

---
**7. Parameters**
- **Definition:** the learned numbers inside a model. "3B" means about 3 billion.
- **Analogy:** the pages of a huge book: more knowledge, heavier to carry.
- **Example:** `llama3.2:3b` ≈ 3 billion; `gpt-oss:20b` ≈ 20 billion.
- **Often confused with:** tokens, the text of each call.
- **Comes back in:** S24, S60.

---
**8. Precision (fp32, fp16, bf16, int8, int4)**
- **Definition:** bits used to store each parameter: fp32 = 4 bytes; fp16/bf16 = 2; int8 = 1; int4 = 0.5.
- **Analogy:** how many digits you write for a price: ₹17,594.0937 or ₹17,594.
- **Example:** a 20B model needs 40 GB at fp16 and 10 GB at int4.
- **Often confused with:** the number of parameters. Precision changes the size of *each* number, not how many.
- **Comes back in:** every local-model decision.

---
**9. Quantization**
- **Definition:** converting weights to lower precision (for example 16-bit to 4-bit): less memory, usually faster, a small accuracy cost.
- **Analogy:** saving a photo as a smaller JPEG: a quarter of the size, almost the same picture.
- **Example:** most default Ollama downloads are already 4-bit.
- **Often confused with:** a smaller model. A quantized 20B model still has 20 billion parameters.
- **Comes back in:** S45, S60.

---
**10. Headroom, activations and the KV cache**
- **Definition:** memory a running model needs *on top of* its weights. **Activations** are each step's working numbers; the **KV cache** remembers what the model has read in this conversation and grows with context length. **Headroom** is the spare memory you leave for these and for everything else.
- **Analogy:** a suitcase needs space to close the lid, not just room for the clothes.
- **Example:** the notebook keeps 2 GB headroom: on 16 GB, at most 14 GB of weights.
- **Comes back in:** S38, S60.

---
**11. RAM vs disk**
- **Definition:** **disk** stores the model file; **RAM** (or GPU memory) is where it must be loaded to run at a usable speed.
- **Analogy:** a book on the shelf vs a book open on your desk.
- **Example:** a 70B model fits on a 1 TB disk, not in 16 GB of RAM.
- **Comes back in:** S60.

---
**12. Tool calling (as a model capability)**
- **Definition:** whether a model, given tools, replies with a **structured tool call** (name + JSON arguments) instead of prose.
- **Analogy:** a lift button: a capable model *presses* it; a weaker one *talks about* pressing it.
- **Example:** `get_account_balance(account_number="00123456")` vs "I would check the balance for you".
- **Often confused with:** an API feature. The API can *offer* tools to any model; only a model trained for it reliably *uses* them.
- **Comes back in:** S20, S34, all of Module 6.

---
**13. Time to first token (TTFT)**
- **Definition:** seconds from sending the request until the first visible word.
- **Analogy:** how long until the waiter brings the first dish.
- **Example:** hosted: usually under a second; a laptop: often several seconds.
- **Often confused with:** total time, which is what a *batch job* feels.
- **Comes back in:** S22 (streaming), S53.

---
**14. Tokens per second (tok/s)**
- **Definition:** output tokens divided by seconds: how fast the answer arrives once it starts.
- **Analogy:** how fast someone talks once they have started.
- **Example:** hosted hardware: hundreds; a laptop: tens. Usually a 5–15× gap; today you measure yours.
- **Comes back in:** S22, S53.

---
**15. Data residency**
- **Definition:** a rule that certain data must stay inside the organisation or the country.
- **Analogy:** documents that may not leave the bank vault.
- **Example:** Session 10's loan-document job (customer PII) must run in-house, which removes every hosted option first.
- **Often confused with:** a preference. It is a **hard constraint**: compliance decides.
- **Comes back in:** S45, S57.

## A4. How this session is built

### The big picture

```
┌──────────────────────── COURSE REPOSITORY (you pull it; you do not edit it) ─────────────────────────┐
│                                                                                                      │
│ module01/local_models_catalog.json   THE DATA                                                        │
│    6 open-weight models: name (Ollama tag), params_billion, precision (all int4),                    │
│    tool_calling true/false, notes. INDICATIVE numbers, not measurements.                             │
│                                                                                                      │
│ module01/local_models_utils.py   TODAY'S TOOLS                                                       │
│    model_size_gb · fits_on · load_local_catalog · laptop_options                                     │
│    probe_tool_calling (+ PROBE_TOOLS, a fake bank tool) · speed_table                                │
│                                                                                                      │
│ module01/providers_utils.py (S11)    reads your .env; groq_client, ollama_client                     │
│ module01/landscape_utils.py (S10)    time_stream (the stopwatch) · PROMPT (the EMI task)             │
│                                                                                                      │
│ module01/s12_local_models.ipynb  ◄── YOU ARE HERE                                                    │
│ tests/test_s12.py   the checkpoint: 6 offline tests + 2 live tests (skip without Ollama)             │
│ module01/reading_local_models.md · module01/terms_s12.md   pre-read and terms                        │
└──────────────────────────────────────────────────────────────────────────────────────────────────────┘
                              │ the notebook talks to ▼
┌─────────────────────────────── YOUR MACHINE (yours; never committed) ────────────────────────────────┐
│ .env               BASE_URL, API_KEY, MODEL (Groq). Read once, at import time.                       │
│ Ollama app         a small server at http://localhost:11434 (no key, no internet)                    │
│    llama3.2:3b     pulled in Session 9   ·   qwen2.5:7b   pulled for today's pre-read                │
│ your RAM           the number you type into MY_RAM_GB in section 3                                   │
└──────────────────────────────────────────────────────────────────────────────────────────────────────┘
                              │ and over the internet ▼
┌─────────────────────────────────── GROQ (hosted; needs your key) ────────────────────────────────────┐
│ openai/gpt-oss-20b: the same open weights as gpt-oss:20b, on fast hosted hardware                    │
└──────────────────────────────────────────────────────────────────────────────────────────────────────┘
```

Nothing is written to a work repository today (those start in Session 14). Your evidence: the notebook output, four reflections, the green test.

### What happens to one prompt in section 5

```
 the EMI prompt (Session 10)                     ┌── Groq client ───► hosted gpt-oss-20b ───┐
 "Applicant earns Rs 60,000 ..."                 │                                          │
          │                                      │                                          ▼
          ▼                                      │                              chunks of text stream back
   time_stream(client, model) ── starts a ───────┤                              one by one: t=0.0 ... 0.4 ...
   stopwatch, sends the prompt, stream=True      │                                          │
                                                 └── Ollama client ──► 1st local model ─────┤
                                                     (localhost:11434)  on YOUR laptop      │
                                                                                            ▼
                         first chunk arrives  →  TTFT (seconds until the first word)
                         last chunk arrives   →  total seconds
                         usage.completion_tokens / total seconds  →  tokens per second
                                                    │
                                                    ▼
                         results = {"groq": {...}, "ollama": {...}}  →  speed_table(results)
                                                    │
                                                    ▼
                         YOU decide: which of Session 10's jobs can live with the slower one?
```

Same function, same prompt, two clients; only the `base_url` differs (Session 11). That makes the comparison fair.

### The helpers you will call, and what each does

| Helper | Lives in | You give it | It gives back | Used in section |
|---|---|---|---|---|
| `groq_client()` | providers_utils (S11) | nothing (reads `.env`) | an OpenAI-style client for Groq | 1 |
| `ollama_client()` | providers_utils (S11) | nothing | a client for `http://localhost:11434/v1` (key `"ollama"`) | 1 |
| `load_local_catalog()` | local_models_utils | nothing | the 6 catalog rows as a list of dicts | 2, 3 |
| `model_size_gb(params_billion, precision)` | local_models_utils | e.g. `20, "int4"` | weight memory in GB, e.g. `10.0` (rule of thumb) | 3 |
| `fits_on(params_billion, precision, ram_gb, headroom_gb=2.0)` | local_models_utils | size, precision, your RAM | `True` / `False` | 3 (inside `laptop_options`) |
| `laptop_options(ram_gb)` | local_models_utils | your RAM in GB | the catalog rows that fit, with 2 GB headroom | 3 |
| `probe_tool_calling(client, model)` | local_models_utils | a client and a model name | `supports_tool_calling`, the text it said, the tool names it called (or an `error`) | 4 |
| `time_stream(client, model)` | landscape_utils (S10) | a client and a model name | `ttft_s`, `total_s`, `completion_tokens`, `tokens_per_s`, `text` | 5 |
| `speed_table(results)` | local_models_utils | `{label: time_stream result}` | a printable table | 5 |
| `check_answer(text)` | landscape_utils (S10) | the model's answer | did it get the EMI and the ratio right? | 5 (optional cell) |

(`ask` is imported but unused today.) Three **under the hood** cells rebuild key helpers and check they match.

### The order of the sections

| # | Section | Calls a model? | Minutes |
|---|---|---|---|
| 1 | Setup | lists your Ollama models (no generation) | 4 |
| 2 | Where open models come from: Hugging Face | no | 5 |
| 3 | Quantization: what it actually costs in memory | no | 10 |
| 4 | Does a small local model support tool calling? | yes: up to 3 local calls (free, no tokens billed) | 10 |
| 5 | Local vs hosted: measure, don't guess | yes: 1 Groq call + 1 local call (+1 optional local) | 10 |
| 6 | Reflection | no | 5 |

## A5. What you will be able to do, and what you will submit

**By the end of this session you will be able to:**
1. **Trace** one open model across its three names and **find** its licence and gating on the model card.
2. **Calculate** weight memory at fp32, fp16, int8 and int4, and **decide** which catalog models fit your laptop with 2 GB headroom.
3. **Explain** quantization, its trade-off, and what the size estimate leaves out.
4. **Run** a tool-calling probe and **interpret** the result (pressed the button, only talked, or errored).
5. **Measure** TTFT and tokens per second, hosted vs local, and **choose** which of Session 10's three jobs can accept the local numbers.

**What you submit:**

| What | Where | What "done" means |
|---|---|---|
| The checkpoint test | `uv run pytest tests/test_s12.py -rs` | **8 passed** with Ollama running, or **6 passed, 2 skipped** (the live Ollama tests) without it. *A skipped test is not a pass.* |
| Reflections 1–4 | section 6 | one or two lines each, with **your own** numbers |
| Your timing table | paste the `speed_table` output under question 4 | both rows, or say which row was skipped and why |

**Budget:** small. **One** Groq call (short prompt, answer capped at 1,500 tokens; the `tokens` column shows the real number). Ollama calls cost no tokens, only time and battery.

---
# Part B · Hands-on

## 1. Setup

### What and why
The notebook needs the helper files, a client for Groq and a client for your local Ollama. It also asks Ollama **which models you have pulled**, because sections 4 and 5 only test models that are really on your machine. If Groq or Ollama is missing, this cell must not crash: it says so plainly and the rest of the notebook runs.

### The idea
Two providers, one kind of client. From Session 11 you know that a "provider" is just a `base_url`, a key and a model name. Ollama is a provider that happens to live on your own laptop:

```
 groq_client()    →  base_url = https://api.groq.com/openai/v1   key = your API_KEY   (internet)
 ollama_client()  →  base_url = http://localhost:11434/v1        key = "ollama"       (your laptop)
                     └───────────── same OpenAI client library, same methods ─────────────┘
```

**Analogy — two branches of the same bank.** Same forms; only the address on the envelope changes. `localhost` means "this computer"; `11434` is the door Ollama listens at.

### What the next cell does
1. **`sys.path[:0] = [...]`** puts the course folders at the *front* of Python's search list (`[:0]` is a slice meaning "before the first item"), so the imports work from the repository root or from `module01/`.
2. **Three `from ... import ...` lines** load today's tools, Session 11's clients and Session 10's stopwatch and prompt. Importing `providers_utils` also **reads your `.env`** (`load_dotenv(override=True)`).
3. **`groq = groq_client() if ... else None`** is a *conditional expression*: "a Groq client **if** there is a real key, **else** `None`". The placeholder `paste_your_groq_key_here` counts as no key. With a real key, nothing changes.
4. **`GROQ_MODEL = os.getenv("MODEL")`** reads the pinned model from `.env` (for the course: `openai/gpt-oss-20b`).
5. **`try: ... except Exception as e:`** *attempts* something that may fail and catches the error instead of crashing. Here: make the Ollama client and ask for its model list.
6. **`[m.id for m in ollama.models.list().data]`** is a *list comprehension*: "for every model `m` Ollama returned, keep its `id`", a one-line loop that builds a list.
7. If Ollama is not running, **`ollama, local_models = None, []`** sets two names at once (*tuple unpacking*). Sections 4 and 5 check them and skip politely.

<details><summary><b>▸ Read more: why does making the Ollama client never fail, but listing the models can?</b></summary>

`ollama_client()` only *creates an object* holding an address; writing an address on an envelope does not post it. The first real request is `ollama.models.list()`, which calls `http://localhost:11434/v1/models`; if the Ollama app is closed, *that* call fails. So the `try` wraps the list call.

The Groq client is different: the OpenAI library refuses to create a client without an API key at all ("Missing credentials"). That is why the cell checks for the key first.
</details>

In [ ]:
import sys; sys.path[:0] = [".", "module01", "../module01"]
import os
from local_models_utils import model_size_gb, fits_on, laptop_options, load_local_catalog, probe_tool_calling, speed_table
from providers_utils import groq_client, ollama_client, ask                 # importing this reads .env (load_dotenv)
from landscape_utils import time_stream, PROMPT

_key = os.getenv("API_KEY") or ""
groq = groq_client() if _key and "paste_your" not in _key else None         # no key -> None: section 5 skips Groq
GROQ_MODEL = os.getenv("MODEL")
if groq is None:
    print("No API_KEY in .env: the Groq part of section 5 will say 'skipped'. Everything else still runs.")
try:
    ollama = ollama_client()
    local_models = [m.id for m in ollama.models.list().data]               # the models YOU have pulled
    print("Ollama models available:", local_models)
except Exception as e:
    ollama, local_models = None, []                                         # sections 4 and 5 check these and skip
    print("Ollama not reachable:", e)

### What you should see
With a key and Ollama running, one line like this (an illustration, not a real run; your list shows *your* models):
```
Ollama models available: ['qwen2.5:7b', 'llama3.2:3b', 'nomic-embed-text:latest']
```
- **The first name in the list is the one section 5 will time**, so note it.
- `nomic-embed-text` (Session 9) is an **embedding** model, not a chat model; if section 4 probes it, it fails, as expected.
- No key adds `No API_KEY in .env: ...`; no Ollama prints `Ollama not reachable: Connection error.` Neither is an error.

| If you see... | It means... | Do this |
|---|---|---|
| `Ollama not reachable: Connection error.` | the Ollama app is not running | start Ollama (Start menu; llama icon in the tray) or `ollama serve` in a second PowerShell window; re-run |
| `Ollama models available: []` | nothing pulled | `ollama pull llama3.2:3b`, re-run |
| `No API_KEY in .env` but you have one | `.env` misplaced or misspelled | `.env` sits next to `pyproject.toml`; restart the kernel |

### Check yourself
*Question 0 (new) — Without running anything: if you close the Ollama app and re-run only this cell, which line will change, and what will `local_models` contain?*

> **Common mistake:** pulling a new model in PowerShell and expecting the notebook to know. `local_models` is a snapshot taken when this cell ran. Re-run the setup cell after every `ollama pull`.

## 2. Where open models come from: Hugging Face

### What and why
Ollama does not train models — it packages and runs models published on hubs like Hugging Face, the same place Groq's hosted open-weight models originate from. The names differ slightly: a model on Hugging Face as `meta-llama/Llama-3.2-3B-Instruct` becomes `llama3.2:3b` as an Ollama tag, and `openai/gpt-oss-20b` on Groq is the same underlying weights as `gpt-oss:20b` if you pulled it locally.

In a company, **"which model are we running?"** is an audit question: where the weights came from, which version, which licence, even when three systems use three names.

### The idea: same weights, three names
```
            the company that trained it publishes the weights + a model card + a licence
                                            │
                                            ▼
   HUGGING FACE (the library)        meta-llama/Llama-3.2-3B-Instruct      ← passport name
            │                                │
            │ Ollama converts and            │ a hosting provider loads the weights
            │ (usually) quantizes            │ on its own fast hardware
            ▼                                ▼
   OLLAMA on your laptop             a HOSTED PROVIDER (e.g. Groq)
   llama3.2:3b                       its own name for it                    ← nicknames
```

**Analogy — a published recipe.** The recipe (weights) is in a public cookbook (Hugging Face). Ollama is your home kitchen; Groq is a restaurant with a professional stove. Same dish, different speed and price. Before cooking it for a business, read the copyright page (the licence).

**On a model card, look for:** (1) the **licence**: commercial use, restrictions; (2) **gated** or not; (3) **intended use and limits**: languages, tested tasks; (4) **size** in parameters, for section 3.

### What the next cell does
1. **`load_local_catalog()`** reads `module01/local_models_catalog.json` and returns its `"models"` list: six dictionaries, one per model.
2. **`print(...)`** prints the raw list; section 3 turns it into a readable table.

In [ ]:
print(load_local_catalog())

### What you should see
One long line: a list of six dictionaries, starting like this (an illustration, not a real run):
```
[{'id': 'llama32-1b', 'name': 'llama3.2:1b', 'params_billion': 1.0, 'precision': 'int4', 'tool_calling': False, ...
```
**How to read one row:** `name` is the Ollama tag (`llama3.2:3b`); `params_billion` is the size (`3.0` = 3 billion); `precision` is bits per weight (`int4`, the usual Ollama default); `tool_calling` is the catalog's opinion; `notes` is one line of advice.

The catalog says these are **indicative** numbers, not measurements. The two `gpt-oss` rows are the same weights Groq hosts; the 120B one needs a GPU server.

**Optional, 2 minutes:** on huggingface.co/models, search `llama 3.2 3b instruct`, find the licence and the "request access" notice; then compare with `qwen2.5 7b instruct`.

### Check yourself
*Question 1 — If a Hugging Face model page says "gated: request access", what does that tell you about its licence?*

> **Common mistake:** assuming "free to download" means "free to use for anything". Licences differ; in a company, the licence is accepted on behalf of the company, not on a personal account.

## 3. Quantization: what it actually costs in memory

### What and why
Every row below is the same family of maths you saw in Session 10: parameters x bytes-per-parameter. The only new idea is precision — how many bytes each parameter uses.

This one sum answers the first question about any local model: **will it fit?** Skip it and you download gigabytes for nothing, or buy a server twice too big.

### The idea
```
 memory for the weights  =  number of parameters  ×  bytes per parameter

 a 20B model:    20,000,000,000  ×  2 bytes (fp16)    =  40,000,000,000 bytes  =  40 GB
                 20,000,000,000  ×  0.5 byte (int4)   =  10,000,000,000 bytes  =  10 GB
```
"20B" is *billion*, and one GB here is a billion bytes, so the billions cancel: **GB = params in billions × bytes per parameter.** You can do it in your head: a 7B model at int4 is 7 × 0.5 = 3.5 GB.

**Analogy — saving a photo as a smaller JPEG.** A quarter of the size, and to most eyes the same picture; look closely and a few details are softer. For a model, "softer" means slightly lower accuracy on hard questions.

Most models are trained and published at 16-bit (fp16/bf16); the usual Ollama download is 4-bit (int4).

### What the next cell does
1. The header line is an **f-string**: text in `f"..."` with values inside `{...}`. After the colon comes a *format spec*: `:10` = at least 10 characters, left-aligned; `:>12` = 12 wide, **right**-aligned.
2. **`for prec in ("fp32", "fp16", "int8", "int4"):`** loops over a *tuple* (a fixed list in round brackets).
3. **`model_size_gb(20, prec)`** computes a 20B model's weight memory; **`:>12.1f`** prints it right-aligned with one decimal.
4. **The middle column is printed empty** (`{'':>12}` is 12 spaces). Fill it in yourself: how many bytes per parameter does each row use?

In [ ]:
print(f"{'precision':10} {'bytes/param':>12} {'a 20B model':>14}")
for prec in ("fp32", "fp16", "int8", "int4"):
    print(f"{prec:10} {'':>12} {model_size_gb(20, prec):>12.1f} GB")

### What you should see
```
precision   bytes/param    a 20B model
fp32                            80.0 GB
fp16                            40.0 GB
int8                            20.0 GB
int4                            10.0 GB
```
Deterministic. **Each step down halves the memory**; fp16 → int4 is the pre-read's "roughly a quarter".

**What it leaves out:** this is weight memory only, a *floor*. A running model also needs activations and the **KV cache**, which grows with the conversation. Hence the 2 GB headroom in the next cell.

### What the next cell does
1. **`MY_RAM_GB = 16`** — **change this to your laptop's RAM** first (Windows: *Settings → System → About → Installed RAM*).
2. **`laptop_options(MY_RAM_GB)`** keeps the catalog rows whose weights fit in `MY_RAM_GB − 2` GB (it calls `fits_on` for each row).
3. **`for m in fits:`** prints each model that fits: name padded to 22 characters, size from `model_size_gb`, and the catalog's `tool_calling` flag.

**Predict first:** with 16 GB, will `gpt-oss:20b (local)` (10 GB at int4) be on the list? With 8 GB?

In [ ]:
# Which catalog models fit YOUR laptop? Edit this to your real RAM.
MY_RAM_GB = 16
fits = laptop_options(MY_RAM_GB)
print(f"On a {MY_RAM_GB} GB laptop, these fit (with 2 GB headroom for everything else):")
for m in fits:
    print(f"  {m['name']:22} {model_size_gb(m['params_billion'], m['precision']):5.1f} GB   tool calling: {m['tool_calling']}")

### What you should see
With the default `MY_RAM_GB = 16` (deterministic, everybody gets this):
```
On a 16 GB laptop, these fit (with 2 GB headroom for everything else):
  llama3.2:1b              0.5 GB   tool calling: False
  llama3.2:3b              1.5 GB   tool calling: False
  qwen2.5:7b               3.5 GB   tool calling: True
  llama3.1:8b              4.0 GB   tool calling: True
  gpt-oss:20b (local)     10.0 GB   tool calling: True
```
`gpt-oss:120b (local)` is missing: 60 GB does not fit in 14. With 8 GB the 20B model drops off too (10 > 6).

**"Fits" is the minimum, not "runs well".** If VS Code, a browser and Teams also want memory, Windows moves memory to disk and everything crawls.

| If you see... | It means... | Do this |
|---|---|---|
| an empty list | `MY_RAM_GB` is below about 2.5 GB, or you typed MB instead of GB | type GB: `8`, `16`, `32` |
| listed here, but Ollama says it needs more memory | the estimate counts weights only | close programs, or use a smaller model |
| `ollama list` shows a bigger size | some parts stay at higher precision | expected: the table is a floor |

<details><summary><b>▸ Read more: GGUF, "Q4_K_M" and other names you will see</b></summary>

Run `ollama show llama3.2:3b` in PowerShell, or browse quantized files on Hugging Face, and you meet **GGUF** and **Q4_K_M**.

- **GGUF** is a file format for running quantized models on ordinary computers; Ollama uses it.
- **Q4_K_M** and similar names describe *how* a model was quantized: `4` is roughly the bits per weight; the letters name the method. Some parts stay at higher precision, one reason the real file is larger than "params × 0.5".

No need to memorise variants. The rule: **fewer bits, less memory, usually more speed, some quality cost**, and only a test on *your* examples tells you if the loss matters (the "verify" step of Session 10).
</details>

### Under the hood: rebuild `model_size_gb`, `fits_on` and `laptop_options` yourself
The three sizing helpers are about ten lines. Here they are, with a check that they give **exactly** the real answers. No model, no network.

**What the next cell does:**
1. Imports **`BYTES_PER_PARAM`**, the helper's real dictionary (it includes `bf16`, which the earlier cell skipped).
2. **`my_model_size_gb`**: billions × bytes per parameter ÷ a billion. `1_000_000_000` is `1000000000`; the `_` is only for readability.
3. **`my_fits_on`** compares that with `ram_gb - headroom_gb`; `headroom_gb=2.0` is a *default argument*, used when the caller gives none.
4. **`my_laptop_options`** is a list comprehension with an `if`: "keep each catalog row `m` **only if** it fits".
5. Prints the precision table with the bytes column filled in.
6. Compares with the real helpers (5 precisions × 7 sizes, 6 laptops). **`assert`** stops with an error if a comparison is false, so reaching the last line proves they all matched.

In [ ]:
from local_models_utils import BYTES_PER_PARAM                     # the real table the helper uses

def my_model_size_gb(params_billion, precision):
    bytes_total = params_billion * 1_000_000_000 * BYTES_PER_PARAM[precision]
    return round(bytes_total / 1_000_000_000, 2)                   # GB here = a billion bytes

def my_fits_on(params_billion, precision, ram_gb, headroom_gb=2.0):
    return my_model_size_gb(params_billion, precision) <= ram_gb - headroom_gb

def my_laptop_options(ram_gb):
    return [m for m in load_local_catalog() if my_fits_on(m["params_billion"], m["precision"], ram_gb)]

print(f"{'precision':10} {'bytes/param':>12} {'a 20B model':>14}")
for prec, b in BYTES_PER_PARAM.items():
    print(f"{prec:10} {b:>12} {my_model_size_gb(20, prec):>12.1f} GB")

for prec in BYTES_PER_PARAM:
    for size in (1, 3, 7, 8, 20, 70, 120):
        assert my_model_size_gb(size, prec) == model_size_gb(size, prec)
for ram in (4, 8, 16, 32, 64, 128):
    mine = [m["name"] for m in my_laptop_options(ram)]
    assert mine == [m["name"] for m in laptop_options(ram)]
    print(f"{ram:>4} GB laptop -> {len(mine)} fit: {', '.join(mine)}")
print("same answers as the real helpers: yes")

**What you should see** (deterministic): five precision rows with bytes `4.0, 2.0, 2.0, 1.0, 0.5`; six lines such as `   8 GB laptop -> 4 fit: llama3.2:1b, llama3.2:3b, qwen2.5:7b, llama3.1:8b`; then `same answers as the real helpers: yes`. `gpt-oss:120b (local)` appears only from 64 GB (120 × 0.5 = 60 ≤ 62), and the KV cache comes on top, so the catalog sends it to a GPU server.

**The take-away:** "will it fit?" is one multiplication and one comparison; that is why the checkpoint tests for it run offline.

### Check yourself
*Question 2 — Why does the table use int4 sizes for the catalog instead of fp16, when fp16 is what the model was trained at?*

> **Common mistake:** confusing **disk** with **RAM**. "500 GB free, so a 70B model will run" is wrong: it must fit in memory to run at a usable speed.

## 4. Does a small local model support tool calling?

### What and why
Not every small model handles function calling reliably — some will just talk about calling the tool instead of actually emitting a structured call. This matters directly for Module 6: an agent that can't reliably call tools can't be an agent.

So before planning "a private agent on our own hardware", test the model. Today's test is tiny: **one question that obviously needs one tool.** Fail this, and it is not your agent's brain.

### The idea: pressing the button vs talking about it
```
 your code sends:  the question  "What's the balance on account 00123456?"
                   + one tool    get_account_balance(account_number: string)
                                            │
                     ┌──────────────────────┴───────────────────────┐
                     ▼                                              ▼
       PRESSES THE BUTTON  (supports_tool_calling=True)  TALKS ABOUT IT  (supports_tool_calling=False)
       message.tool_calls = [ get_account_balance(       message.content = "I would look up account
                              account_number="00123456") ]                   00123456 for you..."
       → your code can run the lookup and continue       → your code has nothing to run; the agent stops
```
**Analogy — a lift button.** Asked for the 5th floor, one colleague presses **5**; another says, "I would press the button marked 5." Only one moved the lift. Your code moves only on a real press: a structured `tool_calls` entry.

**Why small models struggle.** Tool calling is a skill a model must be *trained* for, with a specific reply format. The catalog's `tool_calling` flag is someone else's opinion; the probe is *your* evidence.

### What the next cell does
1. **`if ollama and local_models:`** runs only if Ollama answered **and** a model is pulled; otherwise `skipped (Ollama not running)`.
2. **`for name in local_models[:3]:`** probes at most the first three models (`[:3]` is a slice: items 0, 1, 2).
3. **`probe_tool_calling(ollama, name)`** sends the balance question plus the `get_account_balance` tool, with `tool_choice="auto"` and `max_tokens=200`, and returns a dict.
4. Prints the name and `supports_tool_calling=True/False`.
5. If False, prints **what the model said instead**. **`r.get('raw_text')`** (`dict.get`) returns `None` instead of crashing when the key is missing; `or ''` turns `None` into an empty string; `[:150]` keeps 150 characters.

Each probe is one local generation of up to 200 tokens: seconds to a minute per model. Free, not instant.

In [ ]:
if ollama and local_models:
    for name in local_models[:3]:
        r = probe_tool_calling(ollama, name)
        print(f"{name:20} supports_tool_calling={r['supports_tool_calling']}")
        if not r["supports_tool_calling"]:
            print("   said instead:", (r.get('raw_text') or '')[:150])
else:
    print("skipped (Ollama not running)")

### What you should see
One or two lines per model (an illustration, not a real run; your models and results will differ):
```
qwen2.5:7b           supports_tool_calling=True
llama3.2:3b          supports_tool_calling=False
   said instead: I can't access real-time account information. However, I can guide you on how to check ...
nomic-embed-text:latest supports_tool_calling=False
   said instead:
```
**How to read it:**
- **`True`**: the model pressed the button *this time*. A good sign, not a guarantee: this is the easiest possible test. Session 20 tests harder cases.
- **`False` + text after `said instead:`**: the classic failure, fluent prose and no call. **Data, not a bug.** The catalog marks the small llama3.2 models as not reliable for tools; `llama3.2:3b` may pass this one easy probe or may not. Either way, do not build an agent on it.
- **`False` + nothing after `said instead:`**: the request itself failed (an embedding model always does). The probe stored the reason under `"error"`: `print(probe_tool_calling(ollama, "<name>"))` shows it.

| If you see... | It means... | Do this |
|---|---|---|
| `skipped (Ollama not running)` | no Ollama, or nothing pulled | fine; the under-the-hood cell below still shows the logic |
| `False` for every model | small or not tool-trained models | expected for small llama3.2; try `qwen2.5:7b` |
| an `error` saying *does not support tools* | the model has no tool format | that is your answer; choose another model |

<details><summary><b>▸ Read more: what exactly is sent, and why `tool_choice="auto"`?</b></summary>

The request is a normal chat request plus a `tools` field: tool descriptions in JSON Schema form. `PROBE_TOOLS` describes one fake bank tool, `get_account_balance(account_number)`. Nothing is looked up; the probe only checks *whether the model asks for the tool*.

`tool_choice="auto"` lets the model choose between a tool call and text: the realistic agent setting, so the honest test. Session 20 completes the loop (run the function, send the result back with its tool-call id).
</details>

### Under the hood: the probe's decision, with a fake model
No Ollama needed. We give the **real** `probe_tool_calling` a *fake client* returning replies we wrote, and compare its verdict with a five-line version of the decision: one reply presses the button, one only talks, one request fails.

**What the next cell does:**
1. **`SimpleNamespace`** (standard `types` module) makes a quick object with attributes: `NS(content="hi").content` is `"hi"`. We use it to copy the shape of a real reply, `reply.choices[0].message.tool_calls`.
2. **`FakeClient`** offers the same path as a real client, `client.chat.completions.create(...)`. **`lambda **kwargs: ...`** is a one-line function that accepts *any* keyword arguments (`**kwargs` collects them into a dict), ignores them and returns our reply. Nothing leaves the notebook.
3. **`BrokenClient`** raises an exception, like a model that rejects the request.
4. **`my_decision`** is the heart of the helper: `getattr(message, "tool_calls", None)` reads the attribute or gives `None`; `bool(...)` makes a non-empty list `True`, and `None` or `[]` `False`.
5. Prints the tool the probe offers, then each verdict and whether mine agrees.

In [ ]:
import json
from types import SimpleNamespace as NS
from local_models_utils import PROBE_TOOLS

class FakeClient:                                           # looks like client.chat.completions.create(...)
    def __init__(self, message):
        self.chat = NS(completions=NS(create=lambda **kwargs: NS(choices=[NS(message=message)])))

class BrokenClient:                                         # the request itself fails
    def __init__(self, error_text):
        def fail(**kwargs):
            raise RuntimeError(error_text)
        self.chat = NS(completions=NS(create=fail))

def my_decision(message):
    called = bool(getattr(message, "tool_calls", None))     # a structured call present?
    return {"supports_tool_calling": called, "raw_text": message.content,
            "tool_calls": [tc.function.name for tc in message.tool_calls] if called else []}

print("The tool the probe offers:\n" + json.dumps(PROBE_TOOLS[0]["function"], indent=2) + "\n")

presses = NS(content="", tool_calls=[NS(function=NS(name="get_account_balance",
                                                    arguments='{"account_number": "00123456"}'))])
talks = NS(content="I would look up account 00123456 for you with the balance tool.", tool_calls=None)
for label, msg in [("presses the button", presses), ("only talks", talks)]:
    real = probe_tool_calling(FakeClient(msg), "fake-model")
    print(f"{label:20} real helper: {real}")
    print(f"{'':20} my version agrees: {real == my_decision(msg)}")

real = probe_tool_calling(BrokenClient("fake-model does not support tools"), "fake-model")
print(f"{'request fails':20} real helper: {real}")

**What you should see** (deterministic): the JSON for `get_account_balance`; `presses the button` → `True` with `tool_calls: ['get_account_balance']`; `only talks` → `False` with the sentence in `raw_text`; both with `my version agrees: True`; and `request fails` → `{'supports_tool_calling': False, 'error': 'RuntimeError: fake-model does not support tools'}`. No `raw_text` key there, hence the empty `said instead:` in the real cell.

**The take-away:** the probe does not judge how *smart* the answer is, only whether a structured `tool_calls` entry exists. That is what an agent loop needs.

### Check yourself
*Question 3 — If your local model fails the tool-calling probe, what are your two options for Module 6's agent labs?*

> **Common mistake:** trusting a fluent answer. "I would look up account 00123456 for you" *sounds* right, but code cannot run a sentence. Check `tool_calls`, not the text. And one `True` is a smoke test, not "agent-ready".

## 5. Local vs hosted: measure, don't guess

### What and why
Same prompt, same checkable-answer question from Session 10, sent to Groq and to your local model. Read your own numbers — they depend on your laptop, not on a published benchmark.

A published speed was measured on someone else's hardware and prompt. Your CPU, GPU (if any), free RAM, even whether the laptop is plugged in, change the result. In Session 10's method, this is **verify**.

### The idea: three numbers, two kinds of waiting
```
 request sent                first word                                         last word
     │◄──────── TTFT ─────────►│◄──────────── the answer streams in ─────────────►│
     │   "how long until I     │   tokens per second = output tokens ÷ seconds    │
     │    see anything?"       │   "how fast does it talk once it starts?"        │
     │◄───────────────────────────────── total time ─────────────────────────────►│
                                        "what a batch job feels"
```
Hosted: TTFT usually under a second, hundreds of tok/s. A laptop: often several seconds, tens of tok/s; the gap is usually 5–15×.

**Analogy — a courier vs walking it over yourself.** The courier (hosted) is faster, but the parcel leaves your building. Walking (local) is slower, but it never leaves your hands. The parcel decides, not speed alone.

### What the next cell does
1. **`results = {}`** starts an empty *dictionary*; each provider that succeeds adds an entry.
2. **`if groq:`** — inside, **`time_stream(groq, GROQ_MODEL)`** sends Session 10's EMI prompt with `stream=True` and times it; `except` prints `groq failed: ...` and the cell continues. No key → `groq skipped (no key in .env)`.
3. **`if ollama and local_models:`** does the same for your **first** local model, `local_models[0]`.
4. **`if results:`** — an empty dict counts as `False`, so the table prints only if something answered.

The local answer may take seconds to a couple of minutes (cap: 1,500 tokens). Do not click Stop.

In [ ]:
results = {}
if groq:
    try:
        results["groq"] = time_stream(groq, GROQ_MODEL)
    except Exception as e:
        print("groq failed:", e)
else:
    print("groq skipped (no key in .env)")                 # no key: skip, do not crash
if ollama and local_models:
    try:
        results["ollama"] = time_stream(ollama, local_models[0])
    except Exception as e:
        print("ollama failed:", e)
else:
    print("ollama skipped (not running)")

if results:
    print(speed_table(results))

### What you should see
A small table. The layout is fixed; the numbers below are **an illustration, not a real run** (they are the made-up numbers from the test file):
```
provider     TTFT s   total s   tokens    tok/s
groq           0.40      1.20      120     95.0
ollama         2.10     14.00      120      8.5
```
**How to read it:**
- **TTFT** first: how long would a customer stare at an empty screen?
- **tok/s**: Groq ÷ Ollama is *your* gap. Inside 5–15×, or outside?
- **`tokens`** differ because models write different lengths, so compare tok/s, not total time.
- A reasoning model (like gpt-oss) thinks before it writes; that hidden thinking is not visible text, so its TTFT can look long even on fast hardware.

| If you see... | It means... | Do this |
|---|---|---|
| `groq skipped (no key in .env)` | no key | add your key to `.env`, restart the kernel |
| `groq failed: Error code: 401 ...` | wrong or revoked key | fix `.env`, restart the kernel |
| `groq failed: Error code: 429 ...` | free-tier rate limit | wait a minute, re-run |
| only one row | one provider skipped or failed | read the line above the table |

### Your turn (optional): were the answers right, and how fast is your second model?
A fast wrong answer is still wrong. Session 10's `check_answer` looks for the correct EMI (₹17,594.09, within 1%) and ratio (42.7%, within 0.6 points). The next cell scores the answers you already have (no new call); with `time_second = True` and two local models, it also times your **second** local model.

**What the next cell does:**
1. **`for label, r in results.items():`** — `.items()` gives each *key* (`"groq"`) with its *value* (the timing dict); prints the score out of 2.
2. **`if time_second and ollama and len(local_models) >= 2:`** — all three must be true; then it times `local_models[1]` and prints both local models. `results.get("ollama") or ...` reuses the first model's timing if you have it (`or` runs the right side only when the left is `None`).

In [ ]:
from landscape_utils import check_answer                    # Session 10: did the answer get the numbers right?

time_second = False          # YOUR TURN: set True to also time your SECOND local model (takes a while on a laptop)

if not results:
    print("skipped (no answers to score: no key and no Ollama)")
for label, r in results.items():
    print(f"{label:8} quality check: {check_answer(r['text'])}")

if time_second and ollama and len(local_models) >= 2:
    try:
        local_pair = {local_models[0][:10]: results.get("ollama") or time_stream(ollama, local_models[0]),
                      local_models[1][:10]: time_stream(ollama, local_models[1])}
        print(speed_table(local_pair))
    except Exception as e:
        print("second local model failed:", e)
elif time_second:
    print("skipped (needs Ollama with at least two models pulled)")

**What you should see:** with nothing to score, `skipped (...)`. Otherwise lines like `groq     quality check: {'emi_ok': True, 'ratio_ok': True, 'score_out_of_2': 2}` (an illustration, not a real run). Small local models often miss the arithmetic: part of the trade-off. A 7B model is usually slower per token than a 3B one on the same laptop, since each token needs more than twice the arithmetic.

### Under the hood: what `time_stream` really measures
`time_stream` is a stopwatch around a streaming request. We hand the **real** `time_stream` two *fake* streaming clients, a "fast server" and a "slow laptop", which `time.sleep` before and between words and never touch the network. Then we rebuild `speed_table` and check it prints the identical table.

**What the next cell does:**
1. **`fake_stream(...)`** is a *generator*: each `yield` hands out one chunk shaped like a real streaming chunk (`chunk.choices[0].delta.content`) and pauses until the next is requested. The last chunk carries only `usage.completion_tokens`, like a real stream.
2. **`FakeStreamClient`** wraps it in the `client.chat.completions.create(...)` shape.
3. The real **`time_stream`** notes the first non-empty chunk (TTFT), the end (total), and divides tokens by total time.
4. **`my_speed_table`** rebuilds the table with the same f-string widths and is compared character for character.
5. The last loop recomputes tok/s by hand: *tokens ÷ seconds*.

About one second to run; decimals vary slightly (real timing), the pattern does not.

In [ ]:
import time
from types import SimpleNamespace as NS

def fake_stream(first_wait, per_token_wait, n_tokens):
    time.sleep(first_wait)                                   # "thinking" before the first visible word
    for i in range(n_tokens):
        if i:
            time.sleep(per_token_wait)
        yield NS(choices=[NS(delta=NS(content=f"word{i} "))], usage=None)
    yield NS(choices=[], usage=NS(completion_tokens=n_tokens))   # the final usage-only chunk

class FakeStreamClient:
    def __init__(self, first_wait, per_token_wait, n_tokens):
        self.chat = NS(completions=NS(create=lambda **kwargs: fake_stream(first_wait, per_token_wait, n_tokens)))

fake = {"server": time_stream(FakeStreamClient(0.05, 0.002, 60), "fake"),
        "laptop": time_stream(FakeStreamClient(0.40, 0.020, 30), "fake")}

def my_speed_table(results):
    lines = [f"{'provider':10} {'TTFT s':>8} {'total s':>9} {'tokens':>8} {'tok/s':>8}"]
    for label, r in results.items():
        lines.append(f"{label:10} {r['ttft_s']:8.2f} {r['total_s']:9.2f} {r['completion_tokens']:8d} {r['tokens_per_s']:8.1f}")
    return "\n".join(lines)

print(speed_table(fake))
print("my table identical to the real one:", my_speed_table(fake) == speed_table(fake))
for label, r in fake.items():
    print(f"{label}: {r['completion_tokens']} tokens / {r['total_s']} s = about {r['completion_tokens'] / r['total_s']:.0f} tok/s")

**What you should see** (an illustration, not a real run; your decimals will differ a little):
```
provider     TTFT s   total s   tokens    tok/s
server         0.05      0.17       60    350.0
laptop         0.40      0.98       30     30.6
my table identical to the real one: True
server: 60 tokens / 0.17 s = about 353 tok/s
laptop: 30 tokens / 0.98 s = about 31 tok/s
```
- **TTFT** ≈ the `first_wait` we chose (0.05 and 0.40 s).
- **tok/s** divides by *total* time, so a long TTFT lowers it too. The hand figure differs slightly because the table uses the unrounded time.

**The take-away:** a benchmark is a stopwatch, a counter and one division, and depends entirely on the machine at the other end. These fake speeds only show the mechanism.

### Check yourself
*Question 4 — Your laptop's tokens/second is probably 5-15x slower than Groq's. For which of the three jobs from Session 10 (FAQ bot, document reader, agent-assist) does that gap matter least, and why?*

> **Common mistake:** measuring once on a cold start. The first local request must load the model into memory, so TTFT looks much worse. Run the cell twice and report the second run.

## 6. Reflection

### What and why
Four short answers turn today's numbers into decisions. Use **your own** output. "It depends" is not an answer; "on my 16 GB laptop, the first token took ... s, so ..." is.

### The idea
Each question is one check from Session 10's method: 1 = **licence** (a hard constraint), 2 = **sizing**, 3 = **capability**, 4 = **scoring** with measured speed.

### What to do
1. Write one or two lines under each number below (double-click to edit, Shift+Enter to finish); for question 4, paste your `speed_table` output.
2. Run the checkpoint in the VS Code terminal (PowerShell), from the course folder:
   ```powershell
   uv run pytest tests/test_s12.py -rs
   ```
   `-rs` lists the reason for any skip.

### What you should see
| Test result | It means... | Do this |
|---|---|---|
| `8 passed` | offline and both live Ollama tests pass | done |
| `6 passed, 2 skipped` | the live tests could not reach Ollama | start Ollama and re-run; a skip is not a pass |
| `test_live_ollama_lists_at_least_one_model` fails | Ollama runs, but no model is pulled | `ollama pull llama3.2:3b` |

### Your answers
Answer questions 1-4 above, one or two lines each.

1.
2.
3.
4.

Then run the checkpoint: `uv run pytest tests/test_s12.py`

### Check yourself
*In one sentence each: (a) why might a bank accept a slower model? (b) why is "it fits" not the same as "it runs well"?*

> **Common mistake:** generic answers copied from the slides. **Your** laptop gave **your** numbers; use them.

---
# Part C · Wrap-up

## C1. Summary
- **Open-weight** models are published (mostly on **Hugging Face**) with a **model card** and a **licence**; some are **gated**. **Ollama** packages and runs them; it trains nothing. Same weights, different names.
- **Weight memory = parameters × bytes per parameter**: a 20B model is 80 / 40 / 20 / 10 GB at fp32 / fp16 / int8 / int4. A floor: the KV cache and activations come on top, so keep headroom.
- **Quantization** stores weights with fewer bits: about a quarter of the memory from 16-bit to 4-bit, at a small, task-dependent accuracy cost.
- **Tool calling is a trained skill.** A model either emits a structured `tool_calls` entry or writes prose about it. One probe is a smoke test.
- **Measure, don't guess:** TTFT is the wait, tok/s the pace; hosted is typically 5–15× faster than a laptop. Same function and prompt, only the client changes.
- **Local is a choice for a job:** right when data must stay in-house or for offline and development work; wrong when a hard latency limit (agent-assist, 3 s) cannot be met.

## C2. The words again

| Term | One-line meaning |
|---|---|
| Open-weight model | Weights can be downloaded and run by anyone the licence allows |
| Hugging Face | The public library where open models are published |
| Model card | The model's label: training, intended use, licence, limits |
| Gated model | You must request access and accept the licence before downloading |
| Ollama | A local tool that packages and serves open models at localhost:11434 |
| Ollama tag | The local nickname, e.g. `llama3.2:3b` |
| Parameters | The learned numbers inside a model ("3B" ≈ 3 billion) |
| Precision | Bits per weight: fp32 (4 bytes), fp16/bf16 (2), int8 (1), int4 (0.5) |
| Quantization | Storing weights with fewer bits: less memory, small accuracy cost |
| Headroom / KV cache | Memory needed on top of the weights; the KV cache grows with context |
| RAM vs disk | The model must fit in memory to run, not just on disk |
| Tool calling | The model returns a structured call instead of prose |
| TTFT | Seconds until the first visible word |
| Tokens per second | How fast the answer is produced once it starts |
| Data residency | A rule that data must stay inside the organisation or country |

## C3. Interview practice

**1. "When would you choose a self-hosted open-weight model over a hosted API?"**
*Model answer:* When a hard constraint forbids sending data to a provider (residency, regulated customer or patient data), when you must work offline, or when volume makes owning hardware cheaper than paying per token. In my lab, Session 10's loan-document reader (customer PII) must run in-house, and I sent the same EMI prompt to Groq and to local Ollama with the same `time_stream`; only the `base_url` changed. *Honest limit:* you now own the hardware and its scaling, and a small local model is slower and usually less accurate.

**2. "What is quantization, and what do you lose?"**
*Model answer:* Storing each weight with fewer bits, for example 4 instead of 16: about a quarter of the memory, usually faster. In my lab a 20B model went from 40 GB (fp16) to 10 GB (int4): from "impossible" to "fits a 16 GB laptop". *Honest limit:* an accuracy cost, usually small but task-dependent; only a test on your own task tells.

**3. "Roughly how much memory does a 7-billion-parameter model need?"**
*Model answer:* Parameters × bytes per parameter: about 14 GB at fp16, 7 GB at int8, 3.5 GB at int4, weights only. In my lab, `laptop_options(8)` showed `qwen2.5:7b` (int4) fits an 8 GB laptop with 2 GB headroom. *Honest limit:* a floor; the KV cache grows with context, and the real file is a little larger.

**4. "Is an open-weight model the same as open source?"**
*Model answer:* No. Open-weight means the weights are downloadable; open source in the strict sense also releases code and usually the training data and recipe under an open licence. In my lab, the Llama page on Hugging Face is gated: you accept Meta's licence first, Session 10's licence check in practice. *Honest limit:* licences vary widely; legal or compliance must approve them.

**5. "How would you check that a model supports tool calling before building an agent on it?"**
*Model answer:* Offer one tool with a question that clearly needs it (`tool_choice="auto"`) and check for a structured `tool_calls` entry rather than prose. In my lab, `probe_tool_calling` did this with a fake `get_account_balance` tool; the under-the-hood cell showed the verdict is just "is `tool_calls` non-empty?". *Honest limit:* one easy probe is a smoke test; agents need harder cases: several tools, parallel calls, errors (Session 20) and a decision test set (Session 34).

## C4. Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `Ollama not reachable: Connection error.` | the Ollama app or service is not running | start Ollama (system tray icon), or `ollama serve` in a second PowerShell window; `ollama list` to confirm; re-run section 1 |
| `ollama serve` says *bind: Only one usage of each socket address ...* | Ollama is already running | nothing to fix: use the running one |
| `ollama` is not recognized in PowerShell | not installed, or terminal opened before install | install from ollama.com; reopen every terminal |
| `Ollama models available: []` | nothing pulled | `ollama pull llama3.2:3b` (and `ollama pull qwen2.5:7b` on good Wi-Fi) |
| `model "..." not found, try pulling it first` | the tag is misspelled or not pulled | `ollama list` shows exact tags; `ollama pull <tag>` |
| Ollama: *model requires more system memory ... than is available* | model plus everything else exceeds RAM | close programs, or use `llama3.2:3b` / `llama3.2:1b` |
| the laptop freezes; cells take many minutes | a model that barely fits, swapping to disk | use a smaller model; `ollama ps` shows what is loaded |
| probe False for every model | small models not trained for tool calling | try `qwen2.5:7b`, or route tool tasks to a hosted model |
| `groq skipped` / `groq failed: Error code: 401` | no key / wrong key in `.env` | fix `.env` (no quotes, no spaces), **Kernel → Restart** |
| `ValueError: unknown precision 'fp8'` | a precision the helper does not know | use one of `fp32`, `fp16`, `bf16`, `int8`, `int4` |
| pytest: `6 passed, 2 skipped` | Ollama was not running during the test | start Ollama and re-run; a skip is not a pass |

## C5. What comes next
**Session 13 — Multimodal and structured output.** An image (a scanned KYC document) goes to a vision-capable model, and the answer is forced into a Pydantic schema: *asking* for JSON vs *guaranteeing* it. It closes Module 1.

**Keep from today:** the sizing sum and the headroom habit; Ollama with `llama3.2:3b` and `qwen2.5:7b` (your free fallback when a hosted free tier hits its limit); your probe result (Module 6 needs it); your timing table.